# 01. Data Extraction: Building the Foundation

## Objective
To fetch robust historical data for Tata Motors (TATAMOTORS.NS) and relevant market indices.

## Scope
- **Target:** Tata Motors (`TATAMOTORS.NS`)
- **Indices:** Nifty 50 (`^NSEI`), Nifty Auto (`^CNXAUTO`)
- **Period:** 5 Years (Jan 1, 2020 - Present)

In [6]:
import yfinance as yf
import pandas as pd
import os
import time

# Configure Plotting
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

# Create Data Directories
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

## 2. Robust Data Fetching
We define a function to handle potential API failures.

In [7]:
def fetch_data(ticker, start='2020-01-01', end=None):
    print(f'Fetching {ticker}...')
    try:
        # Try standard download
        df = yf.download(ticker, start=start, end=end, progress=False)
        
        # Check if empty (sometimes yfinance returns empty df without error)
        if df.empty:
            print(f'Warning: No data for {ticker}. Trying without date range (max).')
            df = yf.download(ticker, period='max', progress=False)
        
        # Check again
        if df.empty:
             # Fallback check for BSE if NSE fails
            if '.NS' in ticker:
                bse_ticker = ticker.replace('.NS', '.BO')
                print(f'NSE failed. Trying BSE: {bse_ticker}')
                df = yf.download(bse_ticker, start=start, end=end, progress=False)
        
        if df.empty:
            raise ValueError(f'Could not fetch data for {ticker}')
            
        # Flatten MultiIndex if present (yfinance v0.2+)
        if isinstance(df.columns, pd.MultiIndex):
            if 'Close' in df.columns.get_level_values(0):
                df.columns = df.columns.get_level_values(0)
            elif df.columns.nlevels > 1 and 'Close' in df.columns.get_level_values(1):
                df.columns = df.columns.get_level_values(1)
                
        print(f'Success: {len(df)} rows loaded.')
        return df
        
    except Exception as e:
        print(f'Error fetching {ticker}: {e}')
        return pd.DataFrame()

## 3. Fetching Target & Indices

In [8]:
import yfinance as yf
import pandas as pd
import numpy as np
import warnings
import socket
import requests.packages.urllib3.util.connection as urllib3_cn
from datetime import datetime

# --- CRITICAL FIX: Force IPv4 ---
# This fixes "DNSError" and "Could not resolve host" by forcing Python to use IPv4
def allowed_gai_family():
    return socket.AF_INET
urllib3_cn.allowed_gai_family = allowed_gai_family
# -------------------------------

# Configuration
warnings.simplefilter(action='ignore', category=FutureWarning)

def fetch_stock_data(ticker, start='2019-01-01', end=None, name=None):
    """
    Robust stock data fetcher with IPv4 forcing and auto_adjust handling.
    """
    if end is None:
        end = datetime.now().strftime('%Y-%m-%d')
    
    display_name = name or ticker
    print(f"\n{'='*60}")
    print(f"  Fetching: {display_name} ({ticker})")
    print(f"  Period: {start} -> {end}")
    print(f"{'='*60}")
    
    # Common parameters for yfinance
    yf_params = {
        "start": start, 
        "end": end, 
        "progress": False, 
        "auto_adjust": True  # Fixes FutureWarning and ensures adjusted prices
    }

    # LEVEL 1: Try primary ticker
    try:
        df = yf.download(ticker, **yf_params)
        if not df.empty:
            # Flatten MultiIndex if present (yfinance v0.2+ feature)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            print(f"  ✅ Level 1 SUCCESS | Rows: {len(df)} | Range: {df.index[0].date()} to {df.index[-1].date()}")
            return df
        else:
            print(f"  ⚠️ Level 1 returned empty DataFrame for {ticker}")
    except Exception as e:
        print(f"  ❌ Level 1 FAILED: {e}")
    
    # LEVEL 2: Try alternate exchange (NSE <-> BSE)
    alt_ticker = ticker.replace('.NS', '.BO') if '.NS' in ticker else ticker.replace('.BO', '.NS')
    try:
        print(f"  🔄 Trying alternate ticker: {alt_ticker}...")
        df = yf.download(alt_ticker, **yf_params)
        if not df.empty:
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            print(f"  ✅ Level 2 SUCCESS (alt: {alt_ticker}) | Rows: {len(df)}")
            return df
        else:
            print(f"  ⚠️ Level 2 returned empty DataFrame")
    except Exception as e:
        print(f"  ❌ Level 2 FAILED: {e}")
    
    # LEVEL 3: Try max period as last resort
    try:
        print(f"  🔄 Trying max period...")
        df = yf.download(ticker, period='max', progress=False, auto_adjust=True)
        if not df.empty:
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            df = df[df.index >= start]  # Filter locally
            print(f"  ✅ Level 3 SUCCESS (max period) | Rows: {len(df)}")
            return df
    except Exception as e:
        print(f"  ❌ Level 3 FAILED: {e}")
    
    print(f"  🚫 ALL LEVELS FAILED for {display_name}")
    return pd.DataFrame()

# --- Execution Block ---

TARGET = 'TMPV.NS' 
INDICES = ['^NSEI', '^CNXAUTO']

# Fetch Tata Motors PV
tata_df = fetch_stock_data(TARGET, name="Tata Motors PV")

# Fetch Indices
nifty_df = fetch_stock_data(INDICES[0], name="NIFTY 50")
auto_df = fetch_stock_data(INDICES[1], name="NIFTY Auto")

# Verification
if tata_df.empty:
    print('\nCRITICAL: Tata Motors data is missing. Please check your internet connection or ticker.')
else:
    print(f"\nSuccess! Data loaded for {TARGET}")
    print(tata_df.tail())


  Fetching: Tata Motors PV (TMPV.NS)
  Period: 2019-01-01 -> 2026-02-18
  ✅ Level 1 SUCCESS | Rows: 85 | Range: 2025-10-15 to 2026-02-17

  Fetching: NIFTY 50 (^NSEI)
  Period: 2019-01-01 -> 2026-02-18
  ✅ Level 1 SUCCESS | Rows: 1759 | Range: 2019-01-02 to 2026-02-17

  Fetching: NIFTY Auto (^CNXAUTO)
  Period: 2019-01-01 -> 2026-02-18
  ✅ Level 1 SUCCESS | Rows: 1744 | Range: 2019-01-01 to 2026-02-17

Success! Data loaded for TMPV.NS
Price            Close        High         Low        Open    Volume
Date                                                                
2026-02-11  384.700012  387.350006  382.100006  382.500000  10933650
2026-02-12  383.450012  386.899994  380.149994  385.000000   8324442
2026-02-13  380.250000  387.500000  374.399994  379.000000  15565860
2026-02-16  377.250000  379.600006  375.100006  378.299988   5004467
2026-02-17  382.850006  383.600006  373.549988  376.100006   7843059


## 4. Initial News Check
Checking if we can fetch recent news metadata.

In [9]:
try:
    t = yf.Ticker(TARGET)
    news = t.news
    print(f'Found {len(news)} news items.')
    if news:
        print(news[0]['title'])
except Exception as e:
    print(f'News fetch failed: {e}')

Found 10 news items.
News fetch failed: 'title'


## 5. Save Raw Data

In [10]:
if not tata_df.empty:
    tata_df.to_csv('../data/raw/tata_motors_prices.csv')
    print('Saved tata_motors_prices.csv')
if not nifty_df.empty:
    nifty_df.to_csv('../data/raw/nifty50_prices.csv')
if not auto_df.empty:
    auto_df.to_csv('../data/raw/nifty_auto_prices.csv')

Saved tata_motors_prices.csv
